# Agentic RAG with LLama3.2
Prerequisites
Install Ollama: Run ollama run llama3.2 in your terminal.
Install Libraries: pip install llama-index llama-index-llms-ollama llama-index-embeddings-huggingface

## Models used
 1.Llama 3.2 - for generating text
 2.BAAI/bge-small-en-v1.5 - for embedding
## The Key Packages
1.llama-index	It is the framework that connects the data, the LLM, the Embedding model, and the Vector Store. It provides the logic for the Router and SummaryIndex.
2.llama-index-llms-ollama	The Connector. Python cannot talk to the Ollama application directly. This package acts as the bridge so your Python script can send prompts to Llama 3.2 running on your machine.
3.llama-index-embeddings-huggingface	The Model Loader. This allows LlamaIndex to download the BAAI embedding model from the Hugging Face Hub and run it locally on your CPU/GPU.
4.SentenceSplitter	 It cuts your long PDF text into smaller, bite-sized "chunks" (Nodes). If you don't do this, the text might be too long for the model to read at once.

In [ ]:
import os
import pickle
import ollama
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import PyPDF2

# ==========================================
# 1. PDF PROCESSING (Same as before)
# ==========================================
def extract_text_from_pdfs(pdf_folder_path):
    all_chunks = []
    if not os.path.exists(pdf_folder_path):
        os.makedirs(pdf_folder_path)
        print(f"Created folder '{pdf_folder_path}'. Please put PDFs here.")
        return []

    files = [f for f in os.listdir(pdf_folder_path) if f.endswith('.pdf')]
    print(f"Found {len(files)} PDF files: {files}")

    for filename in files:
        file_path = os.path.join(pdf_folder_path, filename)
        try:
            with open(file_path, 'rb') as f:
                reader = PyPDF2.PdfReader(f)
                full_text = ""
                for page in reader.pages:
                    text = page.extract_text()
                    if text: full_text += text + "\n"
                
                file_chunks = chunk_text(full_text, chunk_size=200, overlap=20)
                labeled_chunks = [f"[Source: {filename}] {chunk}" for chunk in file_chunks]
                all_chunks.extend(labeled_chunks)
        except Exception as e:
            print(f"Error reading {filename}: {e}")
    return all_chunks

def chunk_text(text, chunk_size=200, overlap=20):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

# ==========================================
# 2. AGENTIC RAG CLASS
# ==========================================
class AgenticRAG:
    def __init__(self, llm_model="llama3.2"):
        self.llm_model = llm_model
        self.index_file = "my_rag_db.index"
        self.meta_file = "my_rag_data.pkl"
        
        print("\n--- INITIALIZING AGENTIC RAG ---")
        print("1. Loading Embedding Model...")
        self.embed_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embedding_dim = self.embed_model.get_sentence_embedding_dimension()
        
        # Load or Create DB
        if os.path.exists(self.index_file) and os.path.exists(self.meta_file):
            print("2. Loading Vector DB from disk...")
            self.index = faiss.read_index(self.index_file)
            with open(self.meta_file, "rb") as f:
                self.documents = pickle.load(f)
        else:
            print("2. No existing DB found. Creating new index.")
            self.index = faiss.IndexFlatL2(self.embedding_dim)
            self.documents = []

    def add_documents(self, docs):
        if not docs: return
        print(f"Indexing {len(docs)} chunks...")
        self.documents.extend(docs)
        embeddings = self.embed_model.encode(docs)
        self.index.add(np.array(embeddings).astype('float32'))
        
        faiss.write_index(self.index, self.index_file)
        with open(self.meta_file, "wb") as f:
            pickle.dump(self.documents, f)
        print("✅ Database Saved!")

    # =========================================================================
    # AGENT TOOLS (The "Brain" Functions)
    # =========================================================================

    def router_agent(self, query):
        """Decides if retrieval is needed."""
        print(f"\n--- [AGENT] Step 1: Routing ---")
        prompt = (f"You are a routing agent. The user asked: '{query}'. "
                  f"Do you need to search external documents (PDFs/Knowledge Base) to answer this? "
                  f"If it is a greeting or general knowledge, say NO. If it requires specific facts, say YES. "
                  f"Answer ONLY 'YES' or 'NO'.")
        
        response = ollama.chat(model=self.llm_model, messages=[{'role': 'user', 'content': prompt}])
        decision = response['message']['content'].strip().upper()
        
        if "YES" in decision:
            print(f"   ---> Decision: RETRIEVE (External Knowledge Required)")
            return "RETRIEVE"
        else:
            print(f"   ---> Decision: CHAT (General Conversation)")
            return "CHAT"

    def retrieval_grader_agent(self, query, docs):
        """Self-Correction: Checks if retrieved docs are actually relevant."""
        print(f"\n--- [AGENT] Step 2: Grading Retrieval Relevance ---")
        relevant_docs = []
        for doc in docs:
            prompt = (f"User Question: {query}\n"
                      f"Retrieved Document: {doc}\n"
                      f"Does this document contain keywords or semantic meaning relevant to the question? "
                      f"Answer ONLY 'YES' or 'NO'.")
            
            response = ollama.chat(model=self.llm_model, messages=[{'role': 'user', 'content': prompt}])
            grade = response['message']['content'].strip().upper()
            
            if "YES" in grade:
                print(f"   ---> Doc Accepted (Relevant)")
                relevant_docs.append(doc)
            else:
                print(f"   ---> Doc Rejected (Irrelevant)")
                
        return relevant_docs

    def hallucination_grader_agent(self, context, answer):
        """Self-Correction: Checks if the answer is grounded in the docs."""
        print(f"\n--- [AGENT] Step 4: Hallucination Check ---")
        prompt = (f"Context: {context}\n"
                  f"Answer: {answer}\n"
                  f"Is the answer fully supported by the context provided? "
                  f"Answer ONLY 'YES' or 'NO'.")
        
        response = ollama.chat(model=self.llm_model, messages=[{'role': 'user', 'content': prompt}])
        grade = response['message']['content'].strip().upper()
        
        if "YES" in grade:
            print(f"   ---> Check Passed: Answer is grounded.")
            return True
        else:
            print(f"   ---> Check Failed: Hallucination detected.")
            return False

    def query_rewriter_agent(self, query):
        """Rewrite query if retrieval failed."""
        print(f"\n--- [AGENT] Query Rewriting ---")
        prompt = (f"The previous search for '{query}' yielded no relevant results. "
                  f"Write a better, optimized search query to find the answer. "
                  f"Output ONLY the new query.")
        
        response = ollama.chat(model=self.llm_model, messages=[{'role': 'user', 'content': prompt}])
        new_query = response['message']['content'].strip()
        print(f"   ---> Rewrote query to: '{new_query}'")
        return new_query

    # =========================================================================
    # CORE MECHANISMS
    # =========================================================================

    def retrieve(self, query, top_k=3):
        print(f"   (Tool: Searching Vector DB...)")
        q_embed = self.embed_model.encode([query])
        D, I = self.index.search(np.array(q_embed).astype('float32'), top_k)
        return [self.documents[i] for i in I[0] if i < len(self.documents)]

    def generate(self, query, context):
        print(f"\n--- [AGENT] Step 3: Generation ---")
        prompt = (f"Context: {context}\n"
                  f"Question: {query}\n"
                  f"Answer the question using only the context provided.")
        response = ollama.chat(model=self.llm_model, messages=[{'role': 'user', 'content': prompt}])
        return response['message']['content']

    # =========================================================================
    # THE AGENTIC LOOP (The Graph)
    # =========================================================================
    def run_agentic_pipeline(self, query):
        current_query = query
        max_retries = 2  # To prevent infinite loops
        
        # 1. ROUTING
        route = self.router_agent(current_query)
        if route == "CHAT":
            return self.generate(query, "General Knowledge")

        # 2. RETRIEVAL LOOP
        for attempt in range(max_retries + 1):
            # A. Retrieve
            raw_docs = self.retrieve(current_query)
            
            # B. Grade (Self-Reflection on Documents)
            filtered_docs = self.retrieval_grader_agent(current_query, raw_docs)
            
            if not filtered_docs:
                # REWRITE QUERY LOOP
                print(f"   [!] No relevant documents found. Attempt {attempt+1}/{max_retries}")
                if attempt < max_retries:
                    current_query = self.query_rewriter_agent(current_query)
                    continue # Loop back to retrieval
                else:
                    return "I could not find relevant information in the documents even after rewriting the query."
            
            # C. Generate
            context_str = "\n".join(filtered_docs)
            draft_answer = self.generate(query, context_str)
            
            # D. Hallucination Check (Self-Reflection on Answer)
            if self.hallucination_grader_agent(context_str, draft_answer):
                return draft_answer
            else:
                print("   [!] Generation not grounded. Retrying generation...")
                # In a real system, we might try generating again with stricter prompts
                # For this demo, we return the draft with a warning
                return f"(Warning: Low confidence verification) {draft_answer}"

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Initialize
    agent = AgenticRAG(llm_model="llama3.2")
    pdf_folder = "my_pdfs"
    
    # 2. Ingest Data
    if len(agent.documents) == 0:
        pdf_chunks = extract_text_from_pdfs(pdf_folder)
        if pdf_chunks:
            agent.add_documents(pdf_chunks)
    else:
        print("Using existing database from disk.")

    # 3. Interactive Loop
    if agent.documents:
        while True:
            user_query = input("\n\n----------------------------------\nEnter your question (or 'exit'): ")
            if user_query.lower() == 'exit': break
            
            final_answer = agent.run_agentic_pipeline(user_query)
            
            print(f"\n>>> FINAL ANSWER:\n{final_answer}")
    else:
        print("No documents found. Please add PDFs to 'my_pdfs' folder.")


--- INITIALIZING AGENTIC RAG ---
1. Loading Embedding Model...
2. Loading Vector DB from disk...
Using existing database from disk.

--- [AGENT] Step 1: Routing ---
   ---> Decision: RETRIEVE (External Knowledge Required)
   (Tool: Searching Vector DB...)

--- [AGENT] Step 2: Grading Retrieval Relevance ---
   ---> Doc Accepted (Relevant)
   ---> Doc Accepted (Relevant)
   ---> Doc Accepted (Relevant)

--- [AGENT] Step 3: Generation ---

--- [AGENT] Step 4: Hallucination Check ---
   ---> Check Passed: Answer is grounded.

>>> FINAL ANSWER:
According to the context provided, "Makar" refers to the zodiac sign of Capricorn, which is the sign that the Sun transitions into on January 14 every year, marking the festival of Makar Sankranti.

--- [AGENT] Step 1: Routing ---
   ---> Decision: RETRIEVE (External Knowledge Required)
   (Tool: Searching Vector DB...)

--- [AGENT] Step 2: Grading Retrieval Relevance ---
   ---> Doc Rejected (Irrelevant)
   ---> Doc Rejected (Irrelevant)
   ---> D

User Query
    │
    ▼
① router_agent()       → YES → retrieve  |  NO → answer directly
    │
    ▼
② retrieve() + retrieval_grader_agent()  ← ↩ loop if all docs rejected
    │  ALL REJECTED → query_rewriter_agent() → rewrite → retry (max 2×)
    │  SOME RELEVANT ↓
    ▼
③ generate()           → draft answer from filtered docs only
    │
    ▼
④ hallucination_grader_agent()
    │  GROUNDED  → return final answer  ✅
    └  HALLUCINATION → return with ⚠️ warning

Agentic vs Advanced RAG in one line
Advanced RAG = linear recipe, math-scored reranking, no retry.
Agentic RAG = feedback loop, LLM-reasoned grading, self-correction at both retrieval and generation stages

Advanced RAG uses Math (Cross-Encoder):
It uses a specific model (ms-marco-MiniLM) to assign a numerical score (e.g., 0.85) to documents.
It creates a sorted list. It doesn't "think" about the content; it just measures similarity.
Agentic RAG uses Reasoning (LLM as Judge):
It asks Llama 3.2: "Does this document actually contain the answer to the user's specific question?"
This allows for much smarter filtering. A document might be mathematically similar (high vector score) but factually useless. The Agentic Grader catches this.

1. The Workflow: Linear vs. Cyclic
### Advanced RAG (Linear Pipeline):
The Logic: It follows a strict recipe. Step 1 → Step 2 → Step 3→ Output.
The Flaw: If Step 2 (Retrieval) finds bad documents, Step 3 (Reranker) filters them out, but the system cannot go back to find better ones. It just proceeds with whatever is left.
Code Structure:
### Advanced RAG moves forward only
queries = generate_expansion()
docs = retrieve(queries)
ranked_docs = rerank(docs)
answer = generate(ranked_docs)

##Agentic RAG (Feedback Loop):
The Logic: It acts like a human. It tries something, checks if it worked, and if not, tries again.
The Upgrade: If the Retrieval Grader says "These docs are irrelevant," the system loops back, rewrites the search query, and searches again.
Code Structure:
# Agentic RAG moves backward if needed
for attempt in range(max_retries):
    docs = retrieve(query)
    if not grader(docs):
        query = rewrite_query() # <--- The Loop
        continue
    answer = generate(docs)
    if not hallucination_check(answer):
        continue
    return answer